In [167]:
from collections import defaultdict
from gensim import corpora, models
from csv import QUOTE_NONE
from pathlib import Path
import os
import datetime as dt
import pandas as pd
import nltk

#nltk.download()

In [168]:
# Load school shooters' texts

shooter_paths = dict()
for csv in (Path(os.path.abspath("")).parents[1] / "data" / "school_shooters").rglob("data.csv"):
    shooter_paths[csv.parents[0].name] = csv

shooter_dfs = dict()
for key, value in shooter_paths.items():
    shooter_dfs[key] = pd.read_csv(value, encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)

shooter_dfs

{'Adam_Lanza':            date                                               text
 0    2010-07-05  My favorite two are Bloody Wednesday (It was s...
 1    2010-07-06  I’m probably the only one here who would like ...
 2    2010-07-06  I had the same reaction. I was wondering “Wher...
 3    2010-10-30  I've been considering posting a topic sort of ...
 4    2010-10-31             That water better had not be flavored!
 ..          ...                                                ...
 211  2012-02-24  That's weird. I know that the Latest Activity ...
 212  2012-02-24  I was recently thinking about how hostage situ...
 213  2012-02-26  "I hate how life-apologists say (or rather, th...
 214  2012-02-28  "http://www.youtube.com/watch?v=De-tIflr4E0&t=...
 215  2012-02-28  "I could go in a hundred different ways with t...
 
 [216 rows x 2 columns],
 'Alvaro_Castillo':          date                                               text
 0         NaN  - No cuss words or profanity is used. I do

In [169]:
from nltk.corpus import stopwords
import re

def preprocess_document(text):

    def process_word(word):
        # Some basic preprocessing...
        # Could maybe implement preprocessing of hashtags as well later

        sw = stopwords.words("english")
        url_re = re.compile(r'(http|www)(\S+)?', re.IGNORECASE)
        punctuation_re = r'[^\w\s]'
        is_url = lambda x: True if url_re.search(x) is not None else False

        word = "urlhyperlink" if is_url(word) else word
        word = re.sub(punctuation_re, "", word)

        if word in sw or (len(word) < 3):
            word = ""

        return word

    text = nltk.word_tokenize(text.lower(), preserve_line=False)

    preprocessed_text = []
    for word in text:
        processed_word = process_word(word)
        if processed_word != "" and processed_word != "urlhyperlink":
            preprocessed_text.append(processed_word)

    return preprocessed_text

In [170]:
# Fetching all texts and filtering / preprocessing
import pprint

preprocessed_texts = []
for df in shooter_dfs.values():
    for text in df["text"]:
        preprocessed_texts.append(preprocess_document(text))

print(preprocessed_texts)

""" cho_df = pd.DataFrame(shooter_dfs["Seung_Hui_Cho"])
cho_texts = cho_df["text"].values """

[['favorite', 'two', 'bloody', 'wednesday', 'significantly', 'better', 'shining', 'even', 'many', 'editing', 'errors', 'stalking', 'laura', 'impressively', 'detailed', 'following', 'based', 'true', 'story', 'movie', 'like', 'fictionallyoriented', 'richard', 'farley', 'version', 'zero', 'hour', 'favorite', 'movie', 'genre'], ['probably', 'one', 'would', 'like', 'bloody', 'wednesday', 'stalking', 'laura', 'completely', 'different', 'definitely', 'see', 'one'], ['reaction', 'wondering', 'going', 'chased', 'hotel', 'teddy', 'bear', 'suddenly', 'blew', 'away', 'originally', 'found', 'movie', 'looking', 'george', 'sodini', 'videos', 'last', 'year', 'one', 'restaurant', 'shooting', 'scene', 'title', 'like', 'pittsburgh', 'shooting', 'caught', 'security', 'cameras'], ['considering', 'posting', 'topic', 'sort', 'indirectly', 'pertaining', 'subject', 'gay', 'bullying', 'ever', 'since', 'tyler', 'clementi', 'killed', 'bit', 'hesitant', 'probably', 'make', 'appear', 'deranged'], ['water', 'better'

' cho_df = pd.DataFrame(shooter_dfs["Seung_Hui_Cho"])\ncho_texts = cho_df["text"].values '

In [171]:
# Remove one-off words

frequency = defaultdict(int)
for text in preprocessed_texts:
    for token in text:
        frequency[token] += 1

preprocessed_texts = [[token for token in text if frequency[token] > 1] for text in preprocessed_texts]

dictionary = corpora.Dictionary(preprocessed_texts)
corpus = [dictionary.doc2bow(text) for text in preprocessed_texts]

In [172]:
from gensim import models

n_topics = 10                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

lda_model = models.LdaModel(corpus, num_topics=n_topics, id2word = dictionary)
lda_model.show_topics(num_words=5)

[(0,
  '0.012*"people" + 0.012*"like" + 0.009*"fuck" + 0.008*"fucking" + 0.008*"would"'),
 (1,
  '0.016*"like" + 0.009*"people" + 0.008*"one" + 0.008*"hate" + 0.008*"love"'),
 (2,
  '0.011*"people" + 0.010*"think" + 0.008*"get" + 0.006*"would" + 0.006*"everything"'),
 (3,
  '0.008*"would" + 0.008*"love" + 0.007*"like" + 0.006*"think" + 0.006*"never"'),
 (4,
  '0.014*"like" + 0.008*"would" + 0.007*"time" + 0.007*"god" + 0.007*"manson"'),
 (5,
  '0.009*"fuck" + 0.009*"would" + 0.008*"people" + 0.007*"think" + 0.007*"school"'),
 (6,
  '0.009*"like" + 0.008*"would" + 0.007*"time" + 0.007*"people" + 0.007*"get"'),
 (7,
  '0.012*"like" + 0.011*"would" + 0.010*"people" + 0.008*"one" + 0.007*"want"'),
 (8,
  '0.017*"people" + 0.009*"fucking" + 0.008*"manson" + 0.008*"like" + 0.007*"school"'),
 (9,
  '0.010*"people" + 0.009*"life" + 0.008*"like" + 0.007*"time" + 0.006*"would"')]

In [174]:
""" 
tfidf = models.TfidfModel(corpus)
lsi_model = models.LsiModel(corpus_tfidf, id2word=dictionary, num_topics=n_topics)
corpus_lsi = lsi_model[corpus_tfidf]

lsi_model.print_topics(n_topics) """

' lsi_model = models.LsiModel(corpus_tfidf, id2word=dictionary, num_topics=n_topics)\ncorpus_lsi = lsi_model[corpus_tfidf]\n\nlsi_model.print_topics(n_topics) '